# Ragas Evaluation — Interactive Notebook

Interactive, cell-by-cell version of `eval/evaluate.py`, intended for exploration and debugging before running the full evaluation as a script. Run from the project root, or adjust the path-setup cell below if running from elsewhere.

**Judge model:** Llama 3.3 70B via Groq's free tier — chosen for independence from the pipeline's Azure OpenAI model (no shared training lineage) and for reliable, fast scoring (a local, CPU-bound small model was found to time out and produce unreliable scores on the Faithfulness metric).

**Prerequisites:**
- `.env` configured with Azure OpenAI credentials (same as the main pipeline)
- `.env` configured with `GROQ_API_KEY` (free, no credit card: https://console.groq.com)
- Ollama running locally with the embedding model pulled:
  ```bash
  ollama pull nomic-embed-text
  ```

## Step 1: Path Setup and Imports

Adds the project root to `sys.path` so `config`, `src`, and `eval` are importable
regardless of the notebook's working directory.

In [ ]:
import sys
from pathlib import Path

# Adjust if this notebook is not in the project root or a direct subfolder
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "config").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from config import settings
from src.pipeline import RAGPipeline
from eval.eval_dataset import EVAL_DATASET

from langchain_groq import ChatGroq
from langchain_ollama import OllamaEmbeddings
from ragas import EvaluationDataset, evaluate
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import AnswerRelevancy, ContextPrecision, Faithfulness
from ragas.run_config import RunConfig
from ragas.run_config import RunConfig

import pandas as pd

## Step 2: Inspect the Evaluation Dataset

Reviewing dataset composition — question count and category breakdown — before running
the pipeline against it.

In [ ]:
print(f"Total questions: {len(EVAL_DATASET)}")

from collections import Counter
print(Counter(item["query_type"] for item in EVAL_DATASET))

print("\n--- Sample entry ---")
EVAL_DATASET[0]

## Step 3: Build the Pipeline

Initializes the same `RAGPipeline` used by `main.py`: hybrid retrieval, reranking, and
Azure `gpt-5-mini` generation. This is the system under evaluation.

In [ ]:
pipeline = RAGPipeline().build()
print("Pipeline ready.")

## Step 4: Single-Question Test

Runs one question through the pipeline in isolation to confirm expected behavior and
inspect the exact fields that will be passed to Ragas, before processing the full
dataset.

In [ ]:
sample = EVAL_DATASET[0]
result = pipeline.ask(sample["question"])

print("Question:", sample["question"])
print("\nAnswer:", result["answer"])
print("\nReference (ground truth):", sample["reference"])
print(f"\nRetrieved {len(result['source_documents'])} chunks:")
for doc in result["source_documents"]:
    print(f"  - {doc.metadata.get('source_file')} | {doc.page_content[:80]}...")

## Step 5: Run the Full Dataset Through the Pipeline

Collects the four fields required by Ragas for each question: `user_input`, `response`,
`retrieved_contexts`, and the corresponding hand-written `reference` answer. This step
issues one Azure `gpt-5-mini` call per question — the only Azure cost incurred by this
notebook.

For an initial pass, `EVAL_DATASET[:5]` can be substituted for the full list below.

In [ ]:
# For a quick first run, swap EVAL_DATASET for EVAL_DATASET[:5]
questions = EVAL_DATASET

rows = []
for i, item in enumerate(questions, 1):
    print(f"[{i}/{len(questions)}] {item['question']}")
    result = pipeline.ask(item["question"])
    rows.append({
        "user_input": item["question"],
        "response": result["answer"],
        "retrieved_contexts": [doc.page_content for doc in result["source_documents"]],
        "reference": item["reference"],
    })

print(f"\nCollected {len(rows)} results.")

## Step 6: Inspect Raw Results Before Scoring

Reviewing results as a DataFrame prior to scoring surfaces issues — empty answers, no
retrieved chunks — before judge LLM calls are spent evaluating them.

In [ ]:
raw_df = pd.DataFrame(rows)
raw_df["num_contexts"] = raw_df["retrieved_contexts"].apply(len)
raw_df[["user_input", "response", "num_contexts"]]

## Step 7: Configure the Ragas Judge (Groq + Local Embeddings)

The judge LLM (Llama 3.3 70B via Groq) is independent from the pipeline's own Azure OpenAI model, avoiding any shared-lineage bias between the model being judged and the model doing the judging. Embeddings for the Answer Relevancy metric still run locally via Ollama, since Groq does not offer an embeddings endpoint and this calculation is cheap enough that local CPU inference is not a bottleneck.

In [ ]:
judge_llm = LangchainLLMWrapper(
    ChatGroq(model=settings.GROQ_JUDGE_MODEL, api_key=settings.GROQ_API_KEY)
)
judge_embeddings = LangchainEmbeddingsWrapper(
    OllamaEmbeddings(model=settings.OLLAMA_EMBEDDING_MODEL, base_url=settings.OLLAMA_BASE_URL)
)

# Quick connectivity check before running full evaluation
test_response = judge_llm.langchain_llm.invoke("Reply with the single word: ready")
print("Groq judge responded:", test_response.content)

## Step 8: Run Ragas Evaluation

- **Faithfulness** — whether the answer is supported by the retrieved context
- **Answer Relevancy** — whether the answer addresses the question asked
- **Context Precision** — whether the retrieved chunks were relevant

Groq serves genuine concurrent requests, so a higher `max_workers` is used here than would be safe with a single local Ollama instance.

In [ ]:
ragas_dataset = EvaluationDataset.from_list(rows)

metrics = [Faithfulness(), AnswerRelevancy(), ContextPrecision()]

# Groq handles real concurrency well, unlike a single local Ollama instance
judge_run_config = RunConfig(timeout=300, max_workers=8)

result = evaluate(
    dataset=ragas_dataset,
    metrics=metrics,
    llm=judge_llm,
    embeddings=judge_embeddings,
    run_config=judge_run_config,
)

scores_df = result.to_pandas()
scores_df

## Step 9: Review Scores

Per-question scores identify which specific questions underperform (e.g., low
faithfulness on a given question indicates hallucination on that topic), rather than
only an aggregate average.

In [ ]:
pd.set_option("display.max_colwidth", 60)
scores_df[["user_input", "faithfulness", "answer_relevancy", "context_precision"]]

In [ ]:
print("=== Average scores ===")
for metric in ["faithfulness", "answer_relevancy", "context_precision"]:
    print(f"{metric}: {scores_df[metric].mean():.3f}")

## Step 10: Break Down Scores by Query Type

Since each question in the dataset is tagged with a `query_type`
(`single_document_faq`, `single_document_rba`, `cross_document`, `precision_test`,
`out_of_scope`), grouping scores by type identifies where the pipeline performs well or
poorly — for example, cross-document questions typically score lower on context
precision than single-document lookups.

In [ ]:
scores_df["query_type"] = [item["query_type"] for item in questions]

scores_df.groupby("query_type")[["faithfulness", "answer_relevancy", "context_precision"]].mean()

## Step 11: Save Results

Results are saved to the same `eval/results/` directory used by the standalone script,
so notebook runs and script runs remain directly comparable over time.

In [ ]:
from datetime import datetime

settings.EVAL_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
out_path = settings.EVAL_RESULTS_DIR / f"eval_notebook_{timestamp}.csv"

scores_df.to_csv(out_path, index=False)
print("Saved to:", out_path)

## Next Steps

- Adjust `chunk_size`, `FAISS_TOP_K`, `RERANK_TOP_N`, or the ensemble `weights` in
  `config/settings.py`, rebuild the index, and re-run this notebook to observe the
  effect on scores.
- Low `context_precision` on `cross_document` questions may indicate the reranker's
  `top_n` should be increased so chunks from both source documents survive to
  generation.
- `eval/evaluate.py` runs the same logic as a single command-line script
  (`python eval/evaluate.py`) for repeatable runs outside the notebook.